This code needs to run with a custom environment as analysis3-unstable does not have NEMOSIS.

In [2]:
import os
import glob
import pandas as pd
from datetime import datetime, timedelta
from nemosis import dynamic_data_compiler, static_table
from concurrent.futures import ThreadPoolExecutor, as_completed


# === CONFIGURATION ===
write_path = 'Y:\\ng72\\ms5578\\time_series'
raw_NEM_cache = 'Y:\\ng72\\ms5578\\nemosis_cache'

In [3]:
import nemosis
nemosis.defaults

<module 'nemosis.defaults' from 'c:\\Users\\mehar\\miniconda3\\envs\\dataview\\Lib\\site-packages\\nemosis\\defaults.py'>

In [ ]:

gen_list = static_table('Generators and Scheduled Loads', raw_NEM_cache)
gen_list = gen_list[gen_list['Fuel Source - Primary'] != 'Battery']
gen_list = gen_list[gen_list['Fuel Source - Primary'] != '-']
gen_list = gen_list[gen_list['Dispatch Type'] == 'Generating Unit']

start_date = datetime(2016, 7, 1)
end_date = datetime(2019, 6, 30)

# === Output directory setup ===
output_dir = os.path.join(write_path, "nem_generation")
os.makedirs(output_dir, exist_ok=True)
for f in glob.glob(f"{output_dir}/*.csv"):
    os.remove(f)

# === Time chunk generator ===
def month_range(start, end):
    current = start
    while current < end:
        next_month = (current.replace(day=1) + timedelta(days=32)).replace(day=1)
        yield current, min(next_month - timedelta(seconds=1), end)
        current = next_month

# === Worker function ===
def process_chunk(gen_list, chunk_start, chunk_end):
    try:
        gen_df = dynamic_data_compiler(
            start_time=chunk_start.strftime("%Y/%m/%d %H:%M:%S"),
            end_time=chunk_end.strftime("%Y/%m/%d %H:%M:%S"),
            table_name='DISPATCHLOAD',
            raw_data_location=raw_NEM_cache,
            select_columns=['SETTLEMENTDATE', 'DUID', 'INITIALMW', 'TOTALCLEARED', 'AGCSTATUS']
        )
    except Exception as e:
        print(f"❌ Failed: {chunk_start} – {chunk_end}: {e}")
        return

    if gen_df.empty:
        print(f"⚠️ Empty: {chunk_start} – {chunk_end}")
        return

    gen_df = gen_df[gen_df['DUID'].isin(gen_list['DUID'])]
    if gen_df.empty:
        print(f"⚠️ No matching DUIDs: {chunk_start} – {chunk_end}")
        return

    gen_df = gen_df.rename(columns={'SETTLEMENTDATE': 'time'})
    gen_df['time'] = pd.to_datetime(gen_df['time'])
    gen_df['TOTALMWh'] = gen_df['INITIALMW'] * 5 / 60
    gen_df['TOTALCLEARED'] = gen_df['TOTALCLEARED'] * 5 / 60
    gen_df = gen_df.drop(columns=['INITIALMW'])
    
    agg_func = {'TOTALMWh': 'sum', 'TOTALCLEARED': 'sum', 'AGCSTATUS': 'min'}
    grouped = gen_df.groupby(['DUID', pd.Grouper(key='time', freq='1h')]).agg(agg_func).reset_index()

    for duid, df_duid in grouped.groupby("DUID"):
        safe_duid = duid.replace("/", "_").replace("\\", "_")
        out_path = os.path.join(output_dir, f"{safe_duid}.csv")

        write_header = not os.path.isfile(out_path) or os.path.getsize(out_path) == 0
        df_duid.to_csv(out_path, mode='a', header=write_header, index=False)

    print(f"✅ Processed chunk: {chunk_start} – {chunk_end}")

# time_chunks = list(month_range(start_date, end_date))
# for s, e in time_chunks:
#     process_chunk(gen_list, s, e)

# === Run threads ===
time_chunks = list(month_range(start_date, end_date))
with ThreadPoolExecutor(max_workers=12) as executor:
    futures = [executor.submit(process_chunk, gen_list, s, e) for s, e in time_chunks]
    for f in as_completed(futures):
        f.result()

INFO: Retrieving static table Generators and Scheduled Loads
INFO: Compiling data for table DISPATCHLOAD
INFO: Compiling data for table DISPATCHLOAD
INFO: Compiling data for table DISPATCHLOAD
INFO: Compiling data for table DISPATCHLOAD
INFO: Compiling data for table DISPATCHLOAD
INFO: Compiling data for table DISPATCHLOAD
INFO: Compiling data for table DISPATCHLOAD
INFO: Compiling data for table DISPATCHLOAD
INFO: Compiling data for table DISPATCHLOAD
INFO: Compiling data for table DISPATCHLOAD
INFO: Compiling data for table DISPATCHLOAD
INFO: Compiling data for table DISPATCHLOAD
INFO: Downloading data for table DISPATCHLOAD, year 2016, month 09
INFO: Downloading data for table DISPATCHLOAD, year 2017, month 03
INFO: Downloading data for table DISPATCHLOAD, year 2017, month 01
INFO: Downloading data for table DISPATCHLOAD, year 2016, month 12
INFO: Downloading data for table DISPATCHLOAD, year 2017, month 05
INFO: Downloading data for table DISPATCHLOAD, year 2017, month 04
INFO: Ret

In [ ]:
# Registration list last uploaded 6/6/25

gen_df = pd.read_csv('/g/data/ng72/ms5578/ID_HW_BARRA/data/raw/gen_details.csv')
gen_df.drop(gen_df.filter(regex="Unname"),axis=1, inplace=True)
gen_df.columns = (
    gen_df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^\w_]', '', regex=True)
    .str.replace('__', '_')
)
gen_df = gen_df.rename(columns={'duid': 'DUID'})
gen_df.to_csv('/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_details.csv',
                index=False)

In [ ]:
sdate, edate = "2016/07/01 00:00:00", "2019/06/30 23:59:59"

dem_df = dynamic_data_compiler(start_time=sdate,
                                   end_time=edate,
                                   table_name='DISPATCHREGIONSUM',
                                   raw_data_location=raw_NEM_cache,
                                   select_columns=['REGIONID', 'SETTLEMENTDATE', 'TOTALDEMAND'])


dem_df = dem_df.rename(columns={'SETTLEMENTDATE': "time"})
dem_df['time'] = pd.to_datetime(dem_df['time'])
dem_df["REGIONID"] = dem_df["REGIONID"].astype("category")

dem_df['TOTALDEMAND'] = dem_df['TOTALDEMAND'].apply(lambda x: x*5/60)

dem_df = dem_df.set_index("time").groupby("REGIONID").resample("1h").sum()

dem_df = dem_df.reset_index()

dem_df.to_csv(f'{write_path}/state_demand.csv',
             index=False)